In [ ]:
import geopandas as gpd
import pandas as pd
from pathlib import Path
from shapely.ops import substring
import gc

gc.collect()  # Clean up memory before starting

# ----------------------------------------------------
# SETTINGS
# ----------------------------------------------------

DEV_MODE = True
TEST_ROADS = [2, 3, 9]

# ----------------------------------------------------
# PATH SETUP
# ----------------------------------------------------

try:
    codes_dir = Path(__file__).resolve().parent
except NameError:
    codes_dir = Path.cwd()

project_root = codes_dir.parent
output_folder = project_root / "output"
output_folder.mkdir(exist_ok=True)

# ----------------------------------------------------
# FUNCTIONS
# ----------------------------------------------------

# Cuts the road geometry to match the segment defined by the excel data.
def cut_segment(row):
    geom = row.geometry

    # Skip invalid cases
    if geom is None or geom.length == 0:
        return None

    road_start = row["road_aet"]
    road_end = row["road_let"]

    seg_start = max(row["aet"], road_start)
    seg_end = min(row["let"], road_end)

    if seg_start >= seg_end:
        return None

    # Convert to relative distance (0 → geom.length)
    rel_start = (seg_start - road_start) / (road_end - road_start)
    rel_end = (seg_end - road_start) / (road_end - road_start)

    start_dist = rel_start * geom.length
    end_dist = rel_end * geom.length

    return substring(geom, start_dist, end_dist)


# ----------------------------------------------------
# LOAD DATA
# ----------------------------------------------------

excel = pd.read_excel(
    r"C:\School\TieDataProject\Vaylavirasto-data-projekti\Data\tiet-2-3-9_condition.xlsx",
    header=0,
    usecols=["tie", "kaista", "ajorata", "aosa", "aet", "let", "anomaly_score"]
)

print(excel.head())

roads = gpd.read_file(
    r"C:\School\TieDataProject\KokoSuomi_Digiroad_R_GeoPackage\KokoSuomi_Digiroad_R_GeoPackage.gpkg",
    layer="DR_LINKKI",
    columns=["TIENUMERO", "TIEOSANRO", "AET", "LET", "geometry"]
)

print("roads row count before dropping NULL roads:", len(roads))

roads = roads.dropna(subset=["TIENUMERO"])

print("roads row count after dropping NULL roads:", len(roads))
print("----------------------------------------------")
print(roads.head())
print("----------------------------------------------")
# Normalize column names
excel = excel.rename(columns=str.lower)
roads = roads.rename(columns=str.lower)

roads = roads.rename(columns={
    "tienumero": "tie",
    "tieosanro": "aosa",
    "aet": "road_aet",
    "let": "road_let",
})

roads = roads[roads.geometry.notnull()]

# ----------------------------------------------------
# FILTER (DEV MODE)
# ----------------------------------------------------

if DEV_MODE:
    excel = excel[excel["tie"].isin(TEST_ROADS)]
    roads = roads[roads["tie"].isin(TEST_ROADS)]

# ----------------------------------------------------
# Filter roads to only include those that are in the excel dataset
# ----------------------------------------------------
print("Filtering roads to only include those in the excel dataset...")

keys = excel[["tie", "aosa"]].drop_duplicates()

roads = roads.merge(keys, on=["tie", "aosa"], how="inner")

print(f"Roads after filtering: {len(roads)}")

# ----------------------------------------------------
# SPLIT SINGLE-LANE AND MULTI-LANE
# -----------------------------------------------------
print("Splitting single-lane and multi-lane records...")
# ajorata = 0 means single lane, > 0 means multi-lane

excelSingleLane = excel[excel["ajorata"] == 0]
excelMultiLane = excel[excel["ajorata"] > 0]

print(f"Single lane records: {len(excelSingleLane)}")
print(f"Multi lane records: {len(excelMultiLane)}")

# ----------------------------------------------------
# MERGE WITH EXCEL DATA
# ----------------------------------------------------
print("Merging roads with excel data...")

merged = roads.merge(excel, on=["tie", "aosa"], how="left")

merged = merged[(merged["let"] >= merged["road_aet"]) & (merged["aet"] <= merged["road_let"])]

print("Total segments:", len(merged))
print("Unique road geometries", merged.geometry.nunique())
print("Unique road numbers: ", merged.tie.nunique())

# ----------------------------------------------------
# CUT GEOMETRIES TO MATCH EXCEL SEGMENTS
# ----------------------------------------------------
print("Cutting geometries into 10m segments...")

# segments = []

# for _, row in merged.iterrows():
#     geom = cut_segment(row)
#     if geom:
#         new_row = row.copy()
#         new_row.geometry = geom
#         segments.append(new_row)

# merged = gpd.GeoDataFrame(segments, crs=roads.crs)

merged["geometry"] = merged.apply(cut_segment, axis=1)

# Drop nulls and duplicates that may have been created by cutting
merged = merged[merged.geometry.notnull()].copy()

merged = merged.drop_duplicates(
    subset=["tie", "aosa", "aet", "let", "geometry"]
)

# ----------------------------------------------------
# SAVE TO GEOPACKAGE
# ----------------------------------------------------

output_path = output_folder / "road_condition_10m.gpkg"
merged.to_file(
    output_path,
    driver="GPKG",
    engine="pyogrio", # Use pyogrio for better performance. Ran into problems with fiona when saving large datasets.
    index=False
)

print(f"Saved to: {output_path}")
print("Total segments:", len(merged))
print(merged.head())
